# Generate Masks

## Setup

In [1]:
import sys
from pathlib import Path
from PIL import Image
import random

# Add the project root to sys.path to allow imports from avm package
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from avm.utils import detect_device, load_info_records
from avm.segment import build_model, MODELS, get_model_config_defaults, normalize_category, create_mask_image

## Constants

In [2]:
IMAGES_DIR = '../data/bg1k_imgs/'  # Contains images named <ID>.png e.g., 0.png, 1.png, ...
INFO_TXT_PATH = '../data/bg1k_info.txt'  # Each line: <image name>\t<category>
OUT_MASKS_DIR = f'../data/bg1k_out_masks/'  # Stores output masks named <ID>.png

# Model config
CONFIG_OVERRIDES: dict[str, int | float] = {
    # "mask_threshold": 0.5,
    # "score_threshold": 0.5,
}
MAX_RECORDS: int | None = 20  # Set to None to load all records
SEED = 42
DEVICE = None  # None -> auto detect via detect_device()

## Load Model

In [3]:
device = DEVICE or detect_device()
model_id = MODELS["sam3"]
model = build_model(model_id, device)

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

## Load Records

In [8]:
records = list(load_info_records(INFO_TXT_PATH))
if MAX_RECORDS is not None:
    records = random.Random(SEED).sample(records, min(MAX_RECORDS, len(records)))

print(f'Loaded {len(records)} records.')
print(f'Example records:')
for i, record in enumerate(records[:3]):
    print(f'{record}')

Loaded 20 records.
Example records:
{'image_id': '654', 'image_name': '654.png', 'category': 'Warm milk sterilization', 'mask_image_name': '654_mask.png'}
{'image_id': '114', 'image_name': '114.png', 'category': 'keyboard', 'mask_image_name': '114_mask.png'}
{'image_id': '25', 'image_name': '25.png', 'category': 'fabric sofa', 'mask_image_name': '25_mask.png'}


## Generate

In [9]:
config = {
    **get_model_config_defaults(model_id),
    **CONFIG_OVERRIDES,
}
print(f'Config: {config}')

Config: {'score_threshold': 0.5, 'mask_threshold': 0.5}


In [ ]:
success = 0
failed = 0
skipped = 0

out_dir = Path(OUT_MASKS_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

try:
    for idx, record in enumerate(records, start=1):
        name = record['image_name']
        mask_name = record['mask_image_name']
        category = record['category']
        image = Image.open(Path(IMAGES_DIR) / name)
        out_path = out_dir / mask_name

        if out_path.exists():
            print(f'[{idx}/{len(records)}] Skipping existing mask: {out_path}')
            skipped += 1
            continue

        mask = model.generate_mask(
            image=image, prompt=normalize_category(category), config=config
        )

        if mask is None:
            print(f'[{idx}/{len(records)}] Failed to generate mask: image={name}, category={category}')
            failed += 1
            continue
        
        success += 1
        mask_image = create_mask_image(image=image, mask=mask)
        mask_image.save(out_path)
        print(f'[{idx}/{len(records)}] Successfully generated mask: {out_path}')
except Exception as e:
    print(f'Error during processing: {e}')
finally:
    print(f'Finished processing records: {success} succeeded, {failed} failed, {skipped} skipped.')

[1/20] Failed to generate mask: image=654.png, category=Warm milk sterilization
[2/20] Successfully generated mask: ../data/bg1k_out_masks/114_mask.png
[3/20] Successfully generated mask: ../data/bg1k_out_masks/25_mask.png
[4/20] Successfully generated mask: ../data/bg1k_out_masks/759_mask.png
[5/20] Successfully generated mask: ../data/bg1k_out_masks/281_mask.png
[6/20] Failed to generate mask: image=250.png, category=Tea table or tea table
[7/20] Successfully generated mask: ../data/bg1k_out_masks/228_mask.png
[8/20] Successfully generated mask: ../data/bg1k_out_masks/142_mask.png
[9/20] Successfully generated mask: ../data/bg1k_out_masks/754_mask.png
[10/20] Successfully generated mask: ../data/bg1k_out_masks/104_mask.png
[11/20] Successfully generated mask: ../data/bg1k_out_masks/692_mask.png
[12/20] Successfully generated mask: ../data/bg1k_out_masks/758_mask.png
[13/20] Failed to generate mask: image=913.png, category=investment collection
[14/20] Successfully generated mask: ../